# 01-Foundations

# 14-Introduction-to-Data-Acquisition



[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)

[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
# --- Global Notebook Setup ---
import os
import sys
import math
import time
import random
import json
import textwrap
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError

# Apply the standard course style for all plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6
})
%config InlineBackend.figure_format = 'retina'  # High-res plots

np.set_printoptions(suppress=True, linewidth=120, precision=4)
warnings.filterwarnings('ignore', category=FutureWarning)
print("Environment initialized.")

### Table of Contents
1. [The Lens: Data as Fuel](#The-Lens:-Data-as-Fuel)
2. [The Language of the Web: HTTP](#The-Language-of-the-Web:-HTTP)
3. [Accessing Structured Data with APIs](#Accessing-Structured-Data-with-APIs)
    - [High-Level Convenience vs. Robustness](#High-Level-Convenience-vs.-Robustness)
    - [Direct API Interaction with `requests`](#Direct-API-Interaction-with-requests)
4. [Extracting Unstructured Data with Web Scraping](#Extracting-Unstructured-Data-with-Web-Scraping)
    - [The Ethics of Scraping: `robots.txt`](#The-Ethics-of-Scraping:-robots.txt)
    - [Scraping Static Sites with `requests` and `BeautifulSoup`](#Scraping-Static-Sites-with-requests-and-BeautifulSoup)
    - [Advanced Scraping: Dealing with JavaScript](#Advanced-Scraping:-Dealing-with-JavaScript-Rendered-Pages)
5. [Data Formats: Beyond CSV](#Data-Formats:-Beyond-CSV)
    - [JSON: The Language of APIs](#JSON:-The-Language-of-APIs)
    - [Parquet: The Standard for Big Data](#Parquet:-The-Standard-for-Big-Data)
6. [Summary](#Summary)
7. [Exercises](#Exercises)

# The Lens

Modern empirical economics relies on novel and up-to-the-minute data. The ability to acquire data directly from its source is a crucial skill. This process falls into two categories:

1.  **Using APIs (Application Programming Interfaces):** The structured, preferred method. An API is a formal contract provided by a data source (like the Federal Reserve or World Bank) that specifies how a programmer can request data in a clean, machine-readable format (usually JSON).

2.  **Web Scraping:** The process of extracting information from unstructured websites that do not provide an API. It involves downloading a web page's raw HTML and parsing it to extract specific information.

This notebook provides a practical guide to both methods, covering the underlying web protocols, the tools for interacting with them, and the challenges of acquiring data from both static and dynamic web pages.

### The Language of the Web: HTTP

The **Hypertext Transfer Protocol (HTTP)** governs communication on the web. Every API call or page visit involves an HTTP request-response cycle.

1.  **Request:** Your client sends a request with a:
    - **Method**: The verb indicating the desired action (e.g., `GET` to retrieve data, `POST` to send data).
    - **URL**: The unique address of the resource.
    - **Headers**: Metadata about the request (e.g., `User-Agent` identifying your client, `Accept` specifying desired content types).
    - **Body** (optional): The data being sent in a `POST` request.
2.  **Response:** The server sends back a response with a:
    - **Status Code**: A three-digit code indicating the outcome (e.g., `200 OK` for success, `404 Not Found` for a client error, `500 Internal Server Error` for a server error).
    - **Headers**: Metadata about the response.
    - **Body**: The requested content (e.g., HTML, JSON, or an image file).

### Accessing Structured Data with APIs
An API is the preferred method for data acquisition. It is robust, structured, and respectful of the data provider.

#### High-Level Convenience vs. Robustness
For common economic data sources, libraries like `pandas-datareader` provide a convenient interface. However, they can be brittle; if the source API changes, the wrapper may break. For robust work, interacting directly with the API using `requests` is often superior as it gives you full control over the request structure and error handling.

#### Direct API Interaction with `requests`
Using the `requests` library to interact with APIs directly involves constructing the request URL, handling authentication (e.g., with an API key), and parsing the JSON response.

> **Security Best Practice:** Never hard-code secrets like API keys in your scripts. Store them as environment variables and access them with `os.environ.get()`. This prevents you from accidentally committing sensitive credentials to a public repository like GitHub.

In [ ]:
API_KEY = os.environ.get("FRED_API_KEY")
if not API_KEY:
    print("Skipping direct API call. Set the FRED_API_KEY environment variable to run.")
else:
    base_url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': 'UNRATE', 'api_key': API_KEY, 'file_type': 'json',
        'observation_start': '2020-01-01', 'observation_end': '2023-12-31'
    }
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()  # Raise exception for bad status codes (4xx or 5xx)
        raw_data = response.json()
        df = pd.DataFrame(raw_data['observations'])[['date', 'value']]
        df = df.astype({'date': 'datetime64[ns]', 'value': 'float64'}).set_index('date')
        print("--- Parsed DataFrame from direct API call ---")
        print(df.tail())
    except requests.exceptions.RequestException as e:
        print(f"> **Note:** HTTP Request failed: {e}")
    except (KeyError, json.JSONDecodeError) as e:
        print(f"> **Note:** Failed to parse JSON response: {e}")


### Extracting Unstructured Data with Web Scraping
When an API is unavailable, we can resort to web scraping: programmatically downloading and parsing a web page's HTML. This is a powerful but often brittle technique.

#### The Ethics of Scraping: `robots.txt`
Before scraping, you must check the site's `robots.txt` file. This file, located at the root of a domain (e.g., `https://en.wikipedia.org/robots.txt`), specifies rules for automated agents. While not legally binding, **disobeying `robots.txt` is a serious ethical breach.** Always be a polite scraper: send requests slowly, identify your bot with a `User-Agent` header, and never overload a server.

In [ ]:
try:
    robots_url = 'https://en.wikipedia.org/robots.txt'
    response = requests.get(robots_url)
    print(f"--- Contents of {robots_url} ---")
    print(response.text[:500] + "...") # Print first 500 characters
except requests.exceptions.RequestException as e:
    print(f"> **Note:** Could not fetch robots.txt: {e}")

#### Scraping Static Sites with `requests` and `BeautifulSoup`
The standard toolkit for simple scraping is:
1.  **`requests`**: To download the full HTML content of a URL.
2.  **`BeautifulSoup`**: To parse messy HTML into a structured, searchable object.

In [ ]:
def scrape_sp500_table():
    """Scrapes the S&P 500 components table from Wikipedia."""
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    # It's good practice to identify your bot with a User-Agent header.
    headers = {'User-Agent': 'Jules-Economic-Analysis-Bot/1.0'}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return print(f"> **Note:** Failed to fetch URL: {e}")

    soup = BeautifulSoup(response.text, 'html.parser')
    # Find the table by its unique CSS ID, which is more robust than finding its caption.
    table = soup.find('table', {'id': 'constituents'})
    if not table:
        return print("> **Note:** Could not find S&P 500 constituents table. Page structure may have changed.")
    
    # Use pandas' built-in HTML table parser, which is highly efficient.
    try:
        df = pd.read_html(str(table))[0]
        print(f"> **Note:** Successfully parsed {len(df)} companies from the table.")
        return df
    except Exception as e:
        return print(f"> **Note:** Failed to parse table with pandas: {e}")

sp500_df = scrape_sp500_table()
if sp500_df is not None:
    print(sp500_df.head())

#### Advanced Scraping: Dealing with JavaScript-Rendered Pages
Many modern websites load their content dynamically using JavaScript. `requests` only gets the initial HTML; it does not execute JavaScript. To scrape these sites, we need to automate a real web browser.

While `selenium` is the classic tool for browser automation, **`playwright`** is a more modern library with a simpler API, better performance, and more reliable auto-waiting mechanisms. For new projects, `playwright` is the recommended choice.

In [ ]:
def scrape_dynamic_quotes_playwright():
    """Uses Playwright to scrape quotes from a JavaScript-powered website."""
    try:
        with sync_playwright() as p:
            # headless=True runs the browser in the background without a visible UI.
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()
            
            url = 'http://quotes.toscrape.com/js/'
            page.goto(url, wait_until='domcontentloaded')
            
            # Playwright's locators have auto-waiting built-in. This is crucial for dynamic sites.
            # It waits for the element to appear before proceeding, up to a timeout.
            page.wait_for_selector('div.quote', timeout=10000)
            print("> **Note:** Dynamic content has loaded.")
            
            soup = BeautifulSoup(page.content(), 'html.parser')
            browser.close()
            
            quotes = []
            for quote_div in soup.find_all('div', class_='quote'):
                quotes.append({
                    'text': quote_div.find('span', class_='text').text,
                    'author': quote_div.find('small', class_='author').text
                })
            
            print(f"> **Note:** Successfully scraped {len(quotes)} quotes.")
            print(pd.DataFrame(quotes))
    except PlaywrightTimeoutError:
        print("> **Note:** Playwright timed out waiting for the page to load.")
    except Exception as e:
        print(f"> **Note:** An error occurred with Playwright. You may need to run 'pip install playwright && playwright install'. Error: {e}")

scrape_dynamic_quotes_playwright()

### Data Formats: Beyond CSV

While CSV is ubiquitous, it is not always the best choice for storing and sharing data. It is untyped, text-based (making it large on disk), and row-oriented (slow for column-based analysis).

#### JSON: The Language of APIs
JSON (JavaScript Object Notation) is the standard format for web APIs. It is hierarchical and flexible, mapping directly to Python dictionaries and lists. Pandas can read and write JSON easily, but complex nested structures often require normalization using `pd.json_normalize`.

#### Parquet: The Standard for Big Data
**Apache Parquet** is a columnar storage format optimized for analytics. It is:
- **Typed:** Preserves data types (unlike CSV).
- **Compressed:** Uses efficient compression algorithms (like Snappy or Gzip), resulting in much smaller files.
- **Columnar:** Allows reading only specific columns from disk, which is a massive performance win for large datasets.

For any serious data analysis workflow in Python, Parquet should be the default storage format.

In [ ]:
# Demonstrating Parquet vs CSV
if sp500_df is not None:
    # Save as CSV
    sp500_df.to_csv('sp500.csv', index=False)
    csv_size = os.path.getsize('sp500.csv')

    # Save as Parquet
    sp500_df.to_parquet('sp500.parquet', index=False)
    parquet_size = os.path.getsize('sp500.parquet')

    print(f"CSV Size:     {csv_size/1024:.2f} KB")
    print(f"Parquet Size: {parquet_size/1024:.2f} KB")
    print(f"Parquet is {(1 - parquet_size/csv_size):.1%} smaller than CSV.")

    # Clean up
    os.remove('sp500.csv')
    os.remove('sp500.parquet')

# Summary

In this chapter, we've expanded our ability to acquire data. We've learned:
- **HTTP Mechanics:** Understanding requests and responses is key to debugging data issues.
- **APIs:** The structured, preferred way to get data. Use `requests` for full control.
- **Scraping:** Use `requests` + `BeautifulSoup` for simple static sites, and `playwright` for dynamic JavaScript sites.
- **Formats:** Move beyond CSV to Parquet for performance and type safety.

With these tools, the entire web becomes a potential dataset for your economic research.

### Exercises

1.  **FRED API:** Register for a FRED API key and set it as an environment variable. Write a function that takes a list of FRED series IDs and returns a single, clean Pandas DataFrame containing all of them. Use it to download data on the 10-Year (`DGS10`) and 2-Year (`DGS2`) Treasury rates and plot the yield curve spread (`10Y - 2Y`).

2.  **Static Scraping:** The website `books.toscrape.com` is a sandbox for scraping. Write a script that scrapes the title, price, and star rating of every book on the first page. Store the results in a Pandas DataFrame.

3.  **Scraping Challenge (Pagination):** Extend your script from the previous exercise to handle pagination. Your script should click the "Next" button and continue scraping until it has collected the data for all books across all pages.

4.  **Dynamic Scraping (Advanced):** The website `toscrape.com` also has an "infinite scroll" page at `http://quotes.toscrape.com/scroll`. Write a script using `playwright` that automatically scrolls down the page multiple times to load more quotes and then scrapes all the loaded quotes.